# Phase 1: foundations

Your tasks: understand the files, implement the scorer, inspect labeled groups, and design validation.

Setup and safe previews are provided. Matching and modeling are yours to implement. Start with **Amazon ML (local)** as the kernel. Run cells in order. Exercise checks are skipped until you enable them; skipped does not mean passed.

In [ ]:
from pathlib import Path
import sys, os

# Find the project from either its root or notebooks folder.
cwd = Path.cwd().resolve()
ROOT = next((p for p in [cwd, *cwd.parents] if (p / 'project_config.json').is_file()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from inside the project folder.')
sys.path.insert(0, str(ROOT / 'src'))
for name in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS'):
    os.environ.setdefault(name, '2')
from business_entity_resolution.workspace import CONFIG, DATASET_DIR, dataset_files, iter_tsv, preview
print('Project:', ROOT)
print('Python:', sys.executable)
print('Seed:', CONFIG['seed'])


## 1. Check available resources
Installed RAM and currently available RAM are different. Keep previews small. The configured 4 GiB target is guidance, not a memory limiter.

In [ ]:
import psutil
memory = psutil.virtual_memory()
print(f'Total RAM: {memory.total / 2**30:.1f} GiB; available: {memory.available / 2**30:.1f} GiB')
for path in dataset_files():
    print(path.name, 'OK' if path.is_file() else 'MISSING')


## 2. Inspect small previews
These are only file heads, not random samples or a valid training subset. Empty fields remain empty strings. Only five rows per file are materialized.

In [ ]:
import pandas as pd
from IPython.display import display
for path in dataset_files():
    print(path.name)
    display(pd.DataFrame(preview(path, limit=CONFIG['preview_rows'])))


**Your notes:**
- Why are these files tab-separated?
- Why might a business have multiple matching Source 2 records?
- How is an absent address different from a conflicting address?

Write your answers below.

Your answers: ...

## 3. Read the structural audit
The setup audit is a convenience. It does not verify cross-file ID integrity or business identity. Do not run an expensive full scan every time you restart the notebook.

In [ ]:
import json
report_path = ROOT / 'reports' / 'data_audit_full.json'
if not report_path.exists():
    report_path = ROOT / 'reports' / 'data_audit_preview.json'
if report_path.exists():
    audit = json.loads(report_path.read_text(encoding='utf-8'))
    print('Audit scope:', audit['scope'])
    display(pd.DataFrame([{'file': Path(r['file']).name, 'rows_scanned': r['rows_scanned']} for r in audit['files']]))
    print('Limitations:', audit['limitations'])
else:
    print('Run scripts/audit_data.py from a terminal first.')


## 4. Your ground-truth parser
Implement a function that converts a blank field to an empty set, and a nonblank field to matching IDs. Check duplicate IDs before set conversion. Decide how malformed IDs should be reported.

In [ ]:
def parse_match_ids(text):
    # TODO: implement; preserve empty lists and detect duplicate/malformed IDs.
    raise NotImplementedError('Your Phase 1 exercise')

# After implementing, try: '', 'S2-123,S3-456', 'S2-123,S2-123'.


## 5. Your entity scorer
Let T be true IDs and P be predicted IDs. Derive TP, FP, FN using sets.

F0.5 = 1.25 TP / (1.25 TP + FP + 0.25 FN).

When both sets are empty, score 1. The official macro score averages over every required Source 1 entity, not classes or pairs.

In [ ]:
def score_entity(true_ids, predicted_ids):
    # TODO: implement the metric yourself.
    raise NotImplementedError('Your Phase 1 exercise')


In [ ]:
from math import isclose
run_score_checks = False  # Change to True after implementing score_entity.
cases = [
    ({'A', 'B'}, {'A', 'B'}, 1.0),
    ({'A', 'B'}, {'A'}, 5 / 6),
    ({'A', 'B'}, {'A', 'B', 'C'}, 5 / 7),
    ({'A', 'B'}, set(), 0.0),
    (set(), set(), 1.0),
    (set(), {'C'}, 0.0),
]
if run_score_checks:
    for truth, prediction, expected in cases:
        actual = score_entity(truth, prediction)
        assert isclose(actual, expected, rel_tol=1e-9, abs_tol=1e-12), (truth, prediction, actual, expected)
    print('All six entity-score examples pass.')
else:
    print('Exercise checks SKIPPED. Implement the function, then enable checks.')


**Next exercise:** implement macro averaging with a check that every required reference has a prediction row. The last three examples together must score 1/3. Predicting empty for all training references must score 123247 / 2206821, approximately 0.05585. Explain why pair accuracy would be misleading.

In [ ]:
# TODO: implement your macro scorer and verify the examples above.


## 6. Inspect 20–30 complete labeled groups
Choose references across both countries, singletons, and several match counts. Obtain their target IDs from ground truth, then stream S2/S3 to collect only those targets. Avoid building a lookup dict of every source record.

| Reference ID | What changed? | Evidence that remained | Potential failure mode |
|---|---|---|---|
| Your example | | | |

In [ ]:
# TODO: gather a small collection of complete labeled examples and inspect it.


## 7. Your validation plan
Write your reasoning before generating split files.

- What is the business group you will keep together?
- What if a target ID is linked to two references?
- How will you preserve countries and match-count buckets?
- Where will unmatched target distractors go?
- How will you exclude held-out identities from supervised training negatives?
- How will you save the seed and split manifest?
- Why does a US/India holdout not directly measure France performance?

Proposed starting proportions are 70/15/15; they have NOT been applied yet.

Your split plan: ...

## Done when
- Your parser handles empty lists and malformed duplicates.
- All six scoring examples pass, plus the macro and empty-prediction checks.
- You can explain five interesting labeled groups.
- Your validation plan prevents the same business identity leaking across splits.

Bring those results back for review before Phase 2. See `docs/PHASE_1.md` for troubleshooting.